# DataSuite — feature tour

A runnable walk-through of **every DataSuite feature**. Full reference: [`docs/manual.md`](../docs/manual.md).

| # | Section | Needs a server? |
|---|---|---|
| 0 | Setup + sample data | no |
| 1 | `FrameworkConfig` — pandas or polars | no |
| 2 | `read_local` — CSV / TXT / Excel / Parquet | no |
| 3 | `local_query` — SQL on DataFrames | no |
| 4 | `export_excel` — formatted workbooks | no |
| 5 | `DB2Connect` / `ImpalaConnect` | `LIVE_DB` |
| 6 | `SSHTunnelConnect` | `LIVE_SSH` |
| 7 | `SFTPDataLoader` — browse + load files | demo folder, or `LIVE_SFTP` |
| 8 | `DataStore` — persistent Parquet + DuckDB | no (DB parts: `LIVE_DB`) |
| 9 | Jupyter magics — `%%save`, `%%query`, `%%dql`, `%%parqstore` | no |
| 10 | Clean-up | no |

**Runs top to bottom with no credentials.** Cells that need a real database or bastion are
skipped unless you switch on the matching `LIVE_*` flag below and set the environment
variables (`DB2_*`, `IMPALA_*`, `BASTION_*` — see manual section 2).

SFTP is demonstrated against a **local folder that stands in for an SFTP server**, so every
SFTP feature actually runs. Set `LIVE_SFTP = True` and fill in `SFTP_SETTINGS` to use a real server.

---
## 0. Setup

In [ ]:
# ── Switches for features that need real servers ────────────────────────────
LIVE_DB = False        # DB2Connect / ImpalaConnect / DataStore.connect("db2"|"impala")
LIVE_SSH = False       # open a real SSH tunnel through the bastion
LIVE_SFTP = False      # use a real SFTP server instead of the local demo folder

SFTP_SETTINGS = dict(              # used only when LIVE_SFTP = True
    hostname="sftp.example.com",
    username="user",
    pkey="/path/to/private_key",
    # bastion_hostname="bastion.example.com", bastion_username="jump", bastion_pkey="/path/to/key",
)
SFTP_DIR = "/data"                 # remote folder to browse when LIVE_SFTP = True

In [ ]:
import os, shutil
import pandas as pd
import polars as pl

from datasuite import (
    FrameworkConfig, read_local, local_query, export_excel,
    DB2Connect, ImpalaConnect, SSHTunnelConnect, SFTPDataLoader,
)
from datasuite.parquet_store import DataStore

# Everything this notebook writes lives in these folders (removed in section 10).
DATA_DIR   = "./tour_data"          # sample input files
REMOTE_DIR = "./tour_remote"        # local folder that plays the SFTP server
STORE_DIR  = "./tour_store"         # DataStore base_dir
CACHE_DIR  = "./tour_sftp_cache"    # SFTPDataLoader cache
OUT_DIR    = "./tour_output"        # Excel exports

for d in (DATA_DIR, REMOTE_DIR, STORE_DIR, CACHE_DIR, OUT_DIR):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

### Sample data

Small, synthetic files in every supported format — including a pipe-delimited `.dat`
file and a Thai `cp874`-encoded CSV.

In [ ]:
regions = ["North", "South", "East", "West"]
customers = pd.DataFrame({
    "cust_id": range(1, 21),
    "name": [f"Customer {i:02d}" for i in range(1, 21)],
    "region": [regions[i % 4] for i in range(20)],
    "segment": ["Retail" if i % 3 else "Corporate" for i in range(20)],
})
customers.to_csv(f"{DATA_DIR}/customers.csv", index=False)
customers.to_csv(f"{DATA_DIR}/customers.dat", index=False, sep="|")       # odd extension, pipe-delimited

# Three monthly sales files → one dataset via a glob later
for month in (1, 2, 3):
    pd.DataFrame({
        "sale_id": range(month * 1000, month * 1000 + 50),
        "cust_id": [(i % 20) + 1 for i in range(50)],
        "month": f"2026-{month:02d}",
        "amount": [round(100 + i * 7.5 + month, 2) for i in range(50)],
    }).to_csv(f"{DATA_DIR}/sales_2026_{month:02d}.csv", index=False)

pd.DataFrame({
    "product_id": [1, 2, 3, 4],
    "product": ["Widget", "Gadget", "Gizmo", "Doohickey"],
    "price": [9.99, 24.5, 3.75, 120.0],
}).to_excel(f"{DATA_DIR}/products.xlsx", sheet_name="Products", index=False)

pd.DataFrame({
    "event_id": range(5000),
    "event_type": ["view", "click", "buy", "view", "view"] * 1000,
    "cust_id": [(i % 20) + 1 for i in range(5000)],
}).to_parquet(f"{DATA_DIR}/events.parquet", index=False)

# Thai text in the Windows Thai code page (cp874)
pd.DataFrame({"province": ["กรุงเทพ", "เชียงใหม่", "ภูเก็ต"], "visits": [120, 45, 30]}) \
  .to_csv(f"{DATA_DIR}/provinces_cp874.csv", index=False, encoding="cp874")

sorted(os.listdir(DATA_DIR))

---
## 1. `FrameworkConfig` — pandas or polars output

Every reader/query function takes `engine="pandas"` (→ `pd.DataFrame`) or `engine="polars"`
(→ `pl.LazyFrame`). `FrameworkConfig.dataframe_engine` sets the default for the whole session;
a per-call `engine=` always wins.

In [ ]:
FrameworkConfig.dataframe_engine = "polars"
print(type(read_local(f"{DATA_DIR}/customers.csv")).__name__)                    # LazyFrame
print(type(read_local(f"{DATA_DIR}/customers.csv", engine="pandas")).__name__)   # per-call override

FrameworkConfig.dataframe_engine = "pandas"   # back to the default for the rest of the tour

---
## 2. `read_local` — read CSV, TXT, Excel, ODS, Parquet

The format comes from the extension (override with `file_type=`). Extra keyword arguments go
straight to the reader — pandas option names for the pandas engine, polars names for polars.

In [ ]:
df_customers = read_local(f"{DATA_DIR}/customers.csv")
df_customers.head()

In [ ]:
# Odd extension + delimiter: force the type and pass the pandas option
read_local(f"{DATA_DIR}/customers.dat", file_type="csv", sep="|").head(3)

In [ ]:
read_local(f"{DATA_DIR}/products.xlsx", sheet_name="Products")

In [ ]:
# Polars engine: Parquet and CSV are scanned lazily — nothing is read until .collect()
lf_events = read_local(f"{DATA_DIR}/events.parquet", engine="polars")
lf_events.group_by("event_type").len().sort("len", descending=True).collect()

In [ ]:
# Open file objects work too — then file_type is required
with open(f"{DATA_DIR}/customers.csv", "rb") as fh:
    print(read_local(fh, file_type="csv").shape)

In [ ]:
# Non-UTF-8 text: pass the encoding
read_local(f"{DATA_DIR}/provinces_cp874.csv", encoding="cp874")

---
## 3. `local_query` — SQL over in-memory DataFrames

Every pandas / polars DataFrame **in the calling scope** is a table named after its variable.

In [ ]:
sales = pd.concat(read_local(f"{DATA_DIR}/sales_2026_{m:02d}.csv") for m in (1, 2, 3))
customers_pl = pl.from_pandas(df_customers)          # polars frames work too

by_region = local_query("""
    SELECT c.region, COUNT(*) AS orders, ROUND(SUM(s.amount), 2) AS revenue
    FROM sales s JOIN customers_pl c USING (cust_id)
    GROUP BY c.region ORDER BY revenue DESC
""")
by_region

In [ ]:
# polars output
local_query("SELECT month, SUM(amount) AS total FROM sales GROUP BY month ORDER BY month",
            engine="polars").collect()

In [ ]:
# Inside a function only that function's locals are visible —
# pass anything else as a keyword argument (keyword = table name).
def top_customers(n):
    return local_query("SELECT cust_id, SUM(amount) AS total FROM s GROUP BY 1 ORDER BY 2 DESC LIMIT " + str(n),
                       s=sales)

top_customers(3)

---
## 4. `export_excel` — multi-sheet, formatted workbooks

One dict per sheet. pandas, polars DataFrame and polars LazyFrame are all accepted.
`pretty_format=True` (default) adds an autofilter, a frozen header, auto column widths and
`#,##0.00` on float columns. Empty frames are fine.

In [ ]:
report = f"{OUT_DIR}/sales_report.xlsx"
export_excel(report, [
    {"sheetname": "By region", "dataframe": by_region},                               # pandas
    {"sheetname": "Events",    "dataframe": lf_events.head(100)},                     # polars LazyFrame
    {"sheetname": "Customers", "dataframe": customers_pl},                            # polars DataFrame
    {"sheetname": "Empty",     "dataframe": pd.DataFrame(columns=["a", "b"])},        # empty is OK
])
export_excel(f"{OUT_DIR}/plain.xlsx", [{"sheetname": "Raw", "dataframe": sales}], pretty_format=False)

pd.read_excel(report, sheet_name=None).keys()

---
## 5. `DB2Connect` / `ImpalaConnect` — database queries

`query()` opens an SSH tunnel when this machine is not a direct-access host
(hostnames listed in `DATASUITE_DIRECT_HOSTS`), connects, fetches in 100k-row chunks, and always closes the
connection and the tunnel. Errors are **raised**, not printed.

Credentials come from `DB2_*`, `IMPALA_*` and `BASTION_*` environment variables (read at import time).

In [ ]:
from datasuite.helpers import is_direct_access_host
from datasuite.db_connect import dbc_info

print("direct-access host:", is_direct_access_host())

# Constructor keywords override settings for THAT object only — dbc_info is not changed
prod = DB2Connect()
test = DB2Connect(db2_server="db2-test.example.com", db2_user="tester")
print("test object :", test.dbc.db2_server)
print("prod object :", prod.dbc.db2_server)
print("defaults    :", dbc_info.db2_server)

# A misspelt keyword is an error, not silently ignored
try:
    DB2Connect(db2_sever="typo")
except TypeError as e:
    print("TypeError:", e)

In [ ]:
if LIVE_DB:
    db2 = DB2Connect()
    df_db2 = db2.query("SELECT * FROM SYSIBM.SYSDUMMY1")                    # pd.DataFrame
    lf_db2 = db2.query("SELECT * FROM SYSIBM.SYSDUMMY1", engine="polars")   # pl.LazyFrame

    impala = ImpalaConnect()
    df_imp = impala.query("SELECT 1 AS one")
    display(df_db2, df_imp)

    # Errors propagate — catch them where you want to carry on
    try:
        db2.query("SELECT * FROM NO_SUCH_TABLE")
    except Exception as e:
        print("query failed:", type(e).__name__, e)
else:
    print("LIVE_DB is False — skipped (set DB2_*/IMPALA_*/BASTION_* env vars and LIVE_DB = True)")

In [ ]:
# Low level: a raw DB-API connection + tunnel — you close both
if LIVE_DB:
    from datasuite.db_connect import open_db2_connection
    conn, tunnel = open_db2_connection()          # or open_db2_connection(test.dbc) for overrides
    try:
        cur = conn.cursor()
        cur.execute("SELECT CURRENT DATE FROM SYSIBM.SYSDUMMY1")
        print(cur.fetchall())
    finally:
        conn.close()
        if tunnel:
            tunnel.close()
else:
    print("LIVE_DB is False — skipped")

---
## 6. `SSHTunnelConnect` — port forwarding through a bastion

Reach any service behind a bastion. A free local port in **8801–8899** is chosen automatically.

| `tunnel_type` | How |
|---|---|
| `"ssh"` | system `ssh -N -L` process (waits until the port is open; raises `ConnectionError` / `TimeoutError` on failure) |
| `"python"` | `sshtunnel.SSHTunnelForwarder` |

In [ ]:
tunnel = SSHTunnelConnect(
    dest_server=os.getenv("DB2_SERVER", "db.internal"), dest_port=50000,
    remote_server=os.getenv("BASTION_SERVER", "bastion.example.com"), remote_server_port=22,
    remote_user=os.getenv("BASTION_USER", "me"), remote_private_key=os.getenv("BASTION_PRIVATE_KEY"),
    tunnel_type="ssh", connect_timeout=30,
)
try:
    print("next free local port:", tunnel.port_selection())
except Exception as e:                     # psutil may need extra rights on some systems
    print("port check unavailable:", e)

if LIVE_SSH:
    tunnel.connect()
    print("forwarding 127.0.0.1:%d → %s:%d" % (tunnel.local_port, tunnel.dest_server, tunnel.dest_port))
    tunnel.close()
else:
    print("LIVE_SSH is False — skipped")

---
## 7. `SFTPDataLoader` — browse and load remote files

With `LIVE_SFTP = False` the loader is pointed at **`./tour_remote`, a local folder that stands in
for the SFTP server** (a small adapter with the same methods as paramiko's SFTP client).
Everything below the next cell is identical for a real server.

In [ ]:
# Build the "remote" folder tree
os.makedirs(f"{REMOTE_DIR}/data/exports", exist_ok=True)
os.makedirs(f"{REMOTE_DIR}/data/archive/2025", exist_ok=True)
for m in (1, 2, 3):
    shutil.copy(f"{DATA_DIR}/sales_2026_{m:02d}.csv", f"{REMOTE_DIR}/data/exports/")
shutil.copy(f"{DATA_DIR}/products.xlsx", f"{REMOTE_DIR}/data/exports/")
shutil.copy(f"{DATA_DIR}/events.parquet", f"{REMOTE_DIR}/data/")
shutil.copy(f"{DATA_DIR}/customers.dat", f"{REMOTE_DIR}/data/customers_export")   # no extension
customers.to_csv(f"{REMOTE_DIR}/data/archive/2025/sales_2025_12.csv", index=False)
with open(f"{REMOTE_DIR}/data/exports/README.pdf", "wb") as f:                    # not a table file
    f.write(b"%PDF-1.4 demo")

In [ ]:
import io
from types import SimpleNamespace


class LocalFolderSFTP:
    """Demo stand-in for paramiko.SFTPClient, backed by a local folder."""

    def __init__(self, root):
        self.root = os.path.abspath(root)

    def _local(self, path):
        return os.path.join(self.root, path.lstrip("/"))

    def listdir_attr(self, path):
        folder = self._local(path)
        return [SimpleNamespace(filename=n, **{k: getattr(os.stat(os.path.join(folder, n)), k)
                                                for k in ("st_mode", "st_size", "st_mtime")})
                for n in os.listdir(folder)]

    def open(self, path, mode="r"):
        class _File(io.BytesIO):
            def prefetch(self):
                pass
        with open(self._local(path), "rb") as f:
            return _File(f.read())

    def get(self, remotepath, localpath):
        shutil.copyfile(self._local(remotepath), localpath)

    def close(self):
        pass


if LIVE_SFTP:
    loader = SFTPDataLoader(**SFTP_SETTINGS, cache_dir=CACHE_DIR)
    loader.connect()
    BASE = SFTP_DIR
else:
    loader = SFTPDataLoader(hostname="sftp.example.com", username="demo", cache_dir=CACHE_DIR)
    loader.sftp = LocalFolderSFTP(REMOTE_DIR)       # demo: skip the network connection
    BASE = "/data"

### Finding files — `list_remote`

Lists **table files** (csv, txt, parquet, xlsx, …) newest first, with a summary line.
Columns: `name, folder, file_type, size, modified, path, size_bytes`.

In [ ]:
loader.list_remote(f"{BASE}/exports")

In [ ]:
# Search every sub-folder by name pattern, sorted by name
loader.list_remote(BASE, pattern="sales_*", recursive=True, sort="name")

In [ ]:
# Only one level down; only Parquet; largest first; times in a given time zone
display(loader.list_remote(BASE, recursive=True, max_depth=1, file_types=["parquet"], sort="size", tz="Asia/Bangkok"))

# Every file type (README.pdf and the extension-less file appear too)
loader.list_remote(f"{BASE}/exports", file_types=None, verbose=False)[["name", "file_type", "size"]]

### Loading files — `read_remote`

The file type comes from the extension (pass `file_type=` when there is none). Results are
cached locally as Parquet for `cache_ttl` seconds (default 1 h) — the second call below reads
from the cache.

In [ ]:
df_jan = loader.read_remote(f"{BASE}/exports/sales_2026_01.csv")
df_jan_again = loader.read_remote(f"{BASE}/exports/sales_2026_01.csv")          # from cache
lf_prod = loader.read_remote(f"{BASE}/exports/products.xlsx", engine="polars")
df_cust = loader.read_remote(f"{BASE}/customers_export", file_type="csv", sep="|")  # no extension
df_fresh = loader.read_remote(f"{BASE}/exports/sales_2026_02.csv", use_cache=False) # always download

print(df_jan.shape, lf_prod.collect().shape, df_cust.shape, df_fresh.shape)

In [ ]:
# Typical loop: find files, then load each one
files = loader.list_remote(BASE, pattern="sales_2026_*.csv", recursive=True, sort="name", verbose=False)
q1 = pd.concat(loader.read_remote(p) for p in files["path"])
q1.groupby("month")["amount"].sum()

---
## 8. `DataStore` — persistent Parquet store with DuckDB SQL

Streams data into chunked Parquet under `base_dir`, registers each dataset as a DuckDB view, and
keeps a catalog so datasets survive kernel restarts.

In [ ]:
# Named connections live in connections.yaml (here: a demo file; ${VAR} is expanded from the environment)
CONN_FILE = f"{STORE_DIR}/connections.yaml"
with open(CONN_FILE, "w") as f:
    f.write("""
impala_prod:
  type: impala
  host: impala.example.com
  port: 21050
  auth_mechanism: LDAP
  user: ${IMPALA_USER}
  password: ${IMPALA_PASS}
  ssh:
    host: ${BASTION_SERVER}
    user: ${BASTION_USER}
    private_key: ${BASTION_PRIVATE_KEY}

db2_prod:
  type: db2
  host: db2.example.com
  database: BLUDB
  user: ${DB2_USER}
  password: ${DB2_PASS}

bigquery:
  type: bigquery
  project: my-gcp-project
""")

store = DataStore(STORE_DIR, connections_file=CONN_FILE)
store.connections()          # passwords are never shown

### Saving local files — `save_file`

In [ ]:
store.save_file("customers", f"{DATA_DIR}/customers.csv")

# Glob: three monthly files → one dataset, small chunks so you can see several part files
store.save_file("sales", f"{DATA_DIR}/sales_2026_*.csv", chunk_size=40)

store.save_file("products", f"{DATA_DIR}/products.xlsx", sheet_name="Products")
store.save_file("events", f"{DATA_DIR}/events.parquet", columns=["event_type", "cust_id"],
                chunk_size=2000, compression="zstd")
store.save_file("customers_pipe", f"{DATA_DIR}/customers.dat", file_type="csv", sep="|")

In [ ]:
# use_polars=True: Polars' fast CSV reader. pandas-style options are translated
# (sep → separator, skiprows → skip_rows, usecols, dtype → schema_overrides).
store.save_file("sales_fast", f"{DATA_DIR}/sales_2026_*.csv", use_polars=True,
                sep=",", skiprows=0, usecols="sale_id,amount", dtype={"sale_id": pl.Int64})

# Non-UTF-8 files fall back to the pandas reader automatically
store.save_file("provinces", f"{DATA_DIR}/provinces_cp874.csv", use_polars=True, encoding="cp874")

### Saving DataFrames, SQL results and SFTP files

In [ ]:
store.save_df("by_region", by_region)                       # pandas
store.save_df("event_counts", lf_events.group_by("event_type").len())   # polars LazyFrame

# SQL over existing datasets, using the store's own DuckDB connection
store.save("big_sales", "SELECT * FROM sales WHERE amount > 400", conn=store.ddb)

# A file straight from SFTP (downloaded to a temp folder, streamed, then deleted)
store.save_remote("products_remote", loader, f"{BASE}/exports/products.xlsx")
store.save_remote("customers_remote", loader, f"{BASE}/customers_export", file_type="csv", sep="|")

In [ ]:
# From a database — built-in "db2"/"impala" use the DB2_* / IMPALA_* env vars and auto-tunnel
if LIVE_DB:
    conn = store.connect("db2")
    store.save("db2_dummy", "SELECT * FROM SYSIBM.SYSDUMMY1", conn=conn)
    store.disconnect("db2")
    # Named connection from connections.yaml (opens its ssh tunnel), then close every tunnel
    # conn = store.connect("impala_prod"); store.save("orders", "SELECT ...", conn=conn); store.disconnect()
else:
    print("LIVE_DB is False — skipped")

### Save modes — `error` (default), `append`, `replace`

`replace` writes the new data first and swaps it in only when the write succeeds.

In [ ]:
try:
    store.save_file("customers", f"{DATA_DIR}/customers.csv")            # mode="error"
except FileExistsError as e:
    print("FileExistsError:", e)

store.save_file("jan_feb", f"{DATA_DIR}/sales_2026_01.csv")
store.save_file("jan_feb", f"{DATA_DIR}/sales_2026_02.csv", mode="append")
print("after append :", store.info("jan_feb")["rows"], "rows,", store.info("jan_feb")["chunks"], "part files")

store.save_file("jan_feb", f"{DATA_DIR}/sales_2026_01.csv", mode="replace")
print("after replace:", store.info("jan_feb")["rows"], "rows")

### Dataset names

Any name works except ones with `/ \ :`, a leading `.` or surrounding spaces.
Names that are not plain SQL identifiers must be double-quoted in SQL.

In [ ]:
store.save_df("q1-sales", sales)
display(store.query('SELECT COUNT(*) AS n FROM "q1-sales"'))

try:
    store.save_df("../outside", sales)
except ValueError as e:
    print("ValueError:", e)

### Querying

In [ ]:
store.query("""
    SELECT c.region, p.n_events, ROUND(SUM(s.amount), 2) AS revenue
    FROM sales s
    JOIN customers c USING (cust_id)
    JOIN (SELECT cust_id, COUNT(*) AS n_events FROM events GROUP BY 1) p USING (cust_id)
    GROUP BY 1, 2 ORDER BY revenue DESC LIMIT 5
""")

In [ ]:
lf = store.query("SELECT * FROM sales", engine="polars")       # pl.LazyFrame
print(lf.filter(pl.col("amount") > 400).collect().shape)

rel = store.sql("SELECT month, COUNT(*) AS n FROM sales GROUP BY 1 ORDER BY 1")   # DuckDB relation
rel.show()

### Managing datasets

In [ ]:
store.list()

In [ ]:
store.info("sales")          # "path" is relative to base_dir, so the store can be moved

In [ ]:
store.drop("customers_pipe")                     # unregister; files stay on disk
store.drop("q1-sales", files=True)               # unregister and delete files
store.reload("sales")                            # re-register a view from disk
sorted(store.list()["name"])

---
## 9. Jupyter magics

`enable_magic()` registers `%%parqstore` plus the shortcuts `%%save`, `%%query` and `%%dql`.
Use `%%` when the cell has a body and `%` when it does not.

> `%save` replaces IPython's built-in `%save` for this session.

In [ ]:
store.enable_magic()

### `%%query` — run SQL on the store

In [ ]:
%%query
SELECT * FROM customers LIMIT 5

In [ ]:
%%query --var df_rev
SELECT c.region, ROUND(SUM(s.amount), 2) AS revenue
FROM sales s JOIN customers c USING (cust_id)
GROUP BY 1 ORDER BY 2 DESC

In [ ]:
%%query --var lf_big --polars
SELECT * FROM sales WHERE amount > 400

In [ ]:
print(type(df_rev).__name__, type(lf_big).__name__)
lf_big.group_by("month").agg(pl.col("amount").sum()).sort("month").collect()

In [ ]:
%%query --create march_only
SELECT * FROM sales WHERE month = '2026-03'

In [ ]:
%%query --persist monthly_totals --mode replace
SELECT month, COUNT(*) AS orders, ROUND(SUM(amount), 2) AS total
FROM sales GROUP BY 1 ORDER BY 1

`--create` makes a temporary in-memory DuckDB table; `--persist` saves Parquet that survives a
restart and appears in `store.list()`.

In [ ]:
print("persisted:", "monthly_totals" in store.list()["name"].values,
      "| in-memory only:", "march_only" not in store.list()["name"].values)

### `%%dql` — magic-duckdb, extended with the same flags

In [ ]:
%%dql
SELECT event_type, COUNT(*) AS n FROM events GROUP BY 1 ORDER BY 2 DESC

In [ ]:
%%dql --var df_seg
SELECT segment, COUNT(*) AS customers FROM customers GROUP BY 1

### `%%save` — SQL, file, DataFrame and SFTP modes

In [ ]:
%%save corporate_sales --mode replace --chunk-size 50 --compression gzip
SELECT s.* FROM sales s JOIN customers c USING (cust_id) WHERE c.segment = 'Corporate'

In [ ]:
%%save sales_magic --file --mode replace
./tour_data/sales_2026_*.csv
# reader options, one per line
usecols=sale_id,cust_id,amount

In [ ]:
%%save products_magic --file --file-type xlsx --mode replace
./tour_data/products.xlsx
sheet_name=Products

In [ ]:
%%save sales_polars --file --polars --mode replace
./tour_data/sales_2026_*.csv
sep=,

In [ ]:
lf_regions = pl.LazyFrame({"region": regions, "manager": ["Ann", "Bo", "Chai", "Dee"]})

In [ ]:
%save regions --df lf_regions --mode replace

In [ ]:
%%save exports_jan --remote loader --mode replace
/data/exports/sales_2026_01.csv

In [ ]:
%%save customers_magic --remote loader --file-type csv --mode replace
/data/customers_export
sep=|

With a live database the same magic streams straight from it:

```
%%save orders --conn db2
SELECT * FROM SCHEMA.ORDERS

%%save orders --conn impala_prod          ← name from connections.yaml
SELECT * FROM db.orders
```

### `%%parqstore` — one magic for save, query and SFTP listing

In [ ]:
%%parqstore --query --var df_corp
SELECT COUNT(*) AS n, ROUND(SUM(amount), 2) AS total FROM corporate_sales

In [ ]:
%%parqstore --save top10 --mode replace
SELECT cust_id, SUM(amount) AS total FROM sales GROUP BY 1 ORDER BY 2 DESC LIMIT 10

In [ ]:
%parqstore --list loader /data/exports

In [ ]:
%parqstore --list loader /data "sales_*" -r --sort name --var remote_sales

In [ ]:
%parqstore --list loader /data --all-files --sort size --limit 3

In [ ]:
remote_sales[["name", "folder", "size"]]

In [ ]:
store.list()[["name", "rows", "chunks", "source", "compression"]]

---
## 10. Clean-up

In [ ]:
store.clear_all(files=True)     # drop every dataset and delete its Parquet files
store.disconnect()              # close any tunnels opened by store.connect()
loader.close()                  # close SFTP / SSH / bastion connections

for d in (DATA_DIR, REMOTE_DIR, STORE_DIR, CACHE_DIR, OUT_DIR):
    shutil.rmtree(d, ignore_errors=True)
print("done")